# ДЗ 5. Первый взгляд на orders.csv в Pandas
Срок: **до воскресенья 22.11, 24:00**.
Сдача: папка `hw05` репозитория `dsa-homework` (`hw05.ipynb`): `git add hw05` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число и вывод для менеджера** (в print или Markdown-ячейке). Менеджер не читает код: вывод отвечает, что видно в данных и что делать.

Инструменты: Pandas с урока 5 и всё, что было раньше. `groupby`, `apply`, `.str`-методы (кроме готовой строки для `amount`) пока не используем. Если берёте непройденный инструмент, объясните его в комментарии.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import pandas as pd
# 1
s = pd.Series([10, 20, 30], index=["a", "b", "c"])
print(s["b"], s.iloc[0])
# 2
a = pd.Series({"Алматы": 5, "Шымкент": 2})
b = pd.Series({"Шымкент": 1, "Астана": 4})
print(a + b)
# 3
df = pd.DataFrame({"x": [1, 2, 3], "y": [4, 5, 6]})
print(df[df["x"] > 1].shape)
# 4
print(type(df["x"]).__name__, type(df[["x"]]).__name__)
# 5
f = df[df["x"] > 1]
print(f.index.tolist(), f.iloc[0, 1])
```

Ваши ответы и объяснения:

1. `20 10` - параметр `index` задает ключи каждому элементу списка (a: 10, b: 20, c: 30). Поэтому `s["b"]`=20. А метод `iloc` дает индексирование на основе целочисленных значений для выбора по позиции элемента. Поэтому `s.iloc[0]`=10. 
2. 
```
Алматы  NaN
 Астана  NaN
 Шымкент 3.0
 ````
- Операция `+` между Series выравнивает значений на основе их соответствующих значений ключа и складывает общие ключи, а для уникальных появляется значение `NaN`. Шымкент есть в обоих объектах: (2 + 1 = 3.0). Итоговые индексы автоматически сортируются по алфавиту (Алматы, Астана, Шымкент)
3. `(2, 2)` - условия `df["x"] > 1` возвращает `[False, True, True]`. Затем получаем `df[[False, True, True]]` и филтрация оставляет только те строки, где условие истинно (`{"x":[2,3], "y":[5,6]}`). Так как после фильтрации осталось 2 строки и 2 столбца, получим `(2, 2)`.
4. `Series DataFrame` - Когда передаем в квадратные скобки просто имя столбца (строку `"x"`), pandas извлекает его как одномерный массив — объект Series. Когда передаем список в `df[["x"]]`, pandas возвращает полноценную двумерную таблицу — объект DataFrame. Функция `type()` возвращает строку вида `<class 'pandas.core.series.Series'>`. А атрибут `.__name__` оставляет только короткое название класса: `Series` и `DataFrame`.
5. `[1, 2] 5` - при фильтрации строки сохраняют свои исходные индексы (`1, 2`) из прошлой таблицы. Таблица f будет выглядит так:

```
   x  y
1  2  5
2  3  6
```
- `f.index` обращается к индексам строк (это 1 и 2, так как строка с индексом 0 была отфильтрована в условиях). Затем метод `.tolist()` превращает объект индекса в обычный список: `[1, 2]`. Метод `.iloc` ищет элементы строго по их текущему порядковому номеру, начиная с нуля. 0 — это первая по счёту строка в таблице f (та, у которой индекс 1), а 1 — это второй по счёту столбец (столбец y). На пересечении первой строки и второго столбца находится число 5.

In [1]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("orders.csv")
# Готовая строка: суммы вида "15 990" записаны текстом. Убираем пробел и переводим в числа (подробно на уроке 6)
df["amount"] = pd.to_numeric(df["amount"].str.replace(" ", ""))
print(df.shape)

(5463, 8)


In [3]:
# Даты записаны текстом "2026-09-01 08:00", поэтому строки можно сравнивать
sep = df["order_date"] < "2026-10"     # сентябрь
octo = ~sep                            # октябрь
paid = df["status"] == "оплачен"
not_test = df["customer_id"] != "TEST"
SH = ["Шымкент", "шымкент", "Shymkent"]
print("сентябрь:", sep.sum(), "| октябрь:", octo.sum())

сентябрь: 2782 | октябрь: 2681


## [База] 1. Паспорт таблицы
Сколько строк и столбцов? Какие типы у столбцов? Сколько пропусков в `delivery_days`? Сколько разных значений в `city`, `category`, `status`, `customer_id`?
Напишите менеджеру три проблемы данных, которые видно уже сейчас, и чем каждая опасна для отчёта о выручке.

In [5]:
# 1
print(f"Строки и столбцы: {df.shape}")     # (строк, столбцов)
print(f"Типы у столбцов:\n{df.dtypes}")    # тип каждого столбца
df.info()
print("В delivery_days непустых 5 303 из 5 463: значит, количество пропусков: 160\n")

print("Количество уникальных значений")
print("- Города:", df["city"].nunique())
print("- Категории:", df["category"].nunique())
print("- Статусы:", df["status"].nunique())
print("- Клиенты:", df["customer_id"].nunique())

Строки и столбцы: (5463, 8)
Типы у столбцов:
order_id           int64
order_date        object
customer_id       object
city              object
category          object
amount           float64
delivery_days    float64
status            object
dtype: object
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5463 entries, 0 to 5462
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   order_id       5463 non-null   int64  
 1   order_date     5463 non-null   object 
 2   customer_id    5463 non-null   object 
 3   city           5463 non-null   object 
 4   category       5463 non-null   object 
 5   amount         5463 non-null   float64
 6   delivery_days  5303 non-null   float64
 7   status         5463 non-null   object 
dtypes: float64(2), int64(1), object(5)
memory usage: 341.6+ KB
В delivery_days непустых 5 303 из 5 463: значит, количество пропусков: 160

Количество уникальных значений
- Города: 14
- Категории

#### Три проблемы  данных:
1. Разные варианты написания городов (`city`). Одни и те же города записаны в разных форматах (например, Алматы, алматы, АЛМАТЫ, Almaty). Из-за этого при построении отчёта выручка одного города может распределяться между несколькими категориями, что приведёт к некорректной аналитике по регионам.
2. Пропущенные значения в `delivery_days`. В столбце обнаружено 160 пропусков. Это затрудняет анализ сроков доставки, расчёт среднего времени доставки и оценку качества логистики.
3. Наличие тестовых заказов (`customer_id = TEST`). Тестовые заказы могут попасть в расчёт общей выручки и привести к завышению реальных финансовых показателей.

## [База] 2. Типичный заказ
Для оплаченных заказов без TEST: среднее, медиана, 25-й и 75-й перцентили суммы.
Какой чек «типичный»? Что честнее показать в отчёте, среднее или медиану, и почему?

In [6]:
# 2
paid_not_test = df[paid & not_test]

print("Средний чек:", paid_not_test["amount"].mean())
print("Медиана:", paid_not_test["amount"].median())
print("25-й перцентиль:", paid_not_test["amount"].quantile(0.25))
print("75-й перцентиль:", paid_not_test["amount"].quantile(0.75))

Средний чек: 14672.391348704346
Медиана: 10300.0
25-й перцентиль: 5880.0
75-й перцентиль: 18690.0


**Вывод:**

Какой чек типичный и что честнее показать в отчёте?
- Я бы выбрал **медиану** (10 300 ₸).

Почему?
- 50% заказов имеют сумму меньше или равную 10 300 ₸.
- Другие 50% заказов имеют сумму больше или равную 10 300 ₸.
- Среднее значение составляет 14 672,39 ₸, что заметно выше медианы.

Это говорит о том, что в данных присутствуют крупные заказы, которые увеличивают средний чек.
- Для отчёта я бы использовал медиану как основной показатель, поскольку она меньше зависит от крупных заказов и лучше отражает типичную сумму покупки. Среднее значение также стоит показывать дополнительно, чтобы понимать общую среднюю сумму заказа.

## [База] 3. Отмены по городам
Доля отменённых заказов: по всем заказам, в Шымкенте, в Алматы (все написания каждого города через `isin`).
Какой город выделяется и во сколько раз?

In [10]:
# 3
# Маска для Алматы
AL = ["Алматы", "алматы", "АЛМАТЫ", " Алматы", "Almaty"]

shymkent = df["city"].isin(SH)
almaty = df["city"].isin(AL)

cancelled = df["status"] == "отменён"

# По всем заказам
all_cancel = cancelled.mean() * 100

# В Шымкенте
sh_cancel = cancelled[shymkent].mean() * 100

# В Алматы
al_cancel = cancelled[almaty].mean() * 100

print("По всем заказам доля отмен составляет:", round(all_cancel, 2), "%")
print("В Шымкенте:", round(sh_cancel, 2), "%")
print("В Алматы:", round(al_cancel, 2), "%")

# Сравним с Алматы
print("Доля отмен в Шымкенте примерно в", round(sh_cancel / al_cancel, 1), "раза выше, чем в Алматы.")
print("По сравнению со всеми городами доля отмен в Шымкенте также почти в", round(sh_cancel / all_cancel), "раза выше.")

По всем заказам доля отмен составляет: 10.23 %
В Шымкенте: 20.4 %
В Алматы: 7.86 %
Доля отмен в Шымкенте примерно в 2.6 раза выше, чем в Алматы.
По сравнению со всеми городами доля отмен в Шымкенте также почти в 2 раза выше.


**Вывод:** Шымкент требует дополнительного внимания со стороны бизнеса. Стоит изучить причины отмен заказов в этом городе, чтобы выявить возможные проблемы с доставкой, обслуживанием или процессом оформления заказов.

## [Стандарт] 4. Октябрь против сентября по двум городам
Выручка оплаченных заказов без TEST за сентябрь и октябрь в Шымкенте и в Алматы, изменение в %.
Дубли пока не убираем. Где падение сильнее и во сколько раз? Что сказать менеджеру?

In [14]:
# 4
sh_sep_rev = df.loc[paid & not_test & sep & shymkent, "amount"].sum()
sh_octo_rev = df.loc[paid & not_test & octo & shymkent, "amount"].sum()
al_sep_rev = df.loc[paid & not_test & sep & almaty, "amount"].sum()
al_octo_rev = df.loc[paid & not_test & octo & almaty, "amount"].sum()

print(f"Выручка в Шымкенте: {sh_sep_rev:,.0f} ₸ за сентябрь | {sh_octo_rev:,.0f} ₸ за октябрь")
print(f"Выручка в Алматы: {al_sep_rev:,.0f} ₸ за сентябрь | {al_octo_rev:,.0f} ₸ за октябрь")

sh_change = (sh_octo_rev - sh_sep_rev) / sh_sep_rev * 100
al_change = (al_octo_rev - al_sep_rev) / al_sep_rev * 100

print(f"Изменение: Шымкент {sh_change:.2f}% | Алматы {al_change:.2f}%")
print("в Шымкенте падение выручки примерно в", round(sh_change / al_change, 1), "раза сильнее, чем в Алматы.")

Выручка в Шымкенте: 4,949,800 ₸ за сентябрь | 4,044,010 ₸ за октябрь
Выручка в Алматы: 15,523,940 ₸ за сентябрь | 14,643,020 ₸ за октябрь
Изменение: Шымкент -18.30% | Алматы -5.67%
в Шымкенте падение выручки примерно в 3.2 раза сильнее, чем в Алматы.


**Вывод:**
- Выручка снизилась в обоих городах, однако в Шымкенте падение значительно сильнее — примерно в 3,2 раза относительно Алматы.
- Стоит обратить особое внимание на Шымкент и дополнительно изучить причины снижения выручки.

## [Стандарт] 5. Исправьте код от AI
AI написал код «отменённые заказы Шымкента». В нём **две** ошибки: одна роняет код, вторая тихо даёт неверное число. Найдите обе и исправьте.
```python
cancel_sh = df[df["city"] == "Шымкент" & df["status"] == "отменён"]
print(len(cancel_sh))
```

In [16]:
# 5
cancel_sh = df[(df["city"].isin(SH)) & (df["status"] == "отменён")]
print(len(cancel_sh))

164


- Первая ошибка — отсутствие скобок вокруг условий при использовании `&`. Это приводит к неправильному порядку выполнения операций и ошибке. Оператор `&` имеет более высокий приоритет, чем оператор сравнения `==`.
- Вторая ошибка — сравнение города только с `"Шымкент"`, хотя в данных есть разные варианты написания. Из-за этого часть заказов не учитывается.

## [Стандарт] 6. Самые частые клиенты
Кто сделал больше всего заказов (`value_counts` по `customer_id`)? Покажите все заказы этого клиента (`loc` с маской, столбцы дата, город, сумма, статус).
Есть ли среди его заказов повторяющиеся `order_id`? Что это значит для выручки, которую мы считали весь урок?

In [24]:
# 6
# Клиент сделавщий больше всего заказов
cust_counts = df["customer_id"].value_counts()
print(f"Клиент сделавщий больше всего заказов:\n{cust_counts.head(1)}\n")


# Все заказы этого клиента
cust_id = cust_counts.index[0]
client_orders = df.loc[df["customer_id"] == cust_id, ["order_id", "order_date", "city", "amount", "status"]]
print(f"Все заказы этого клиента:\n{client_orders}\n")

# Для повторяющие заказов подсчитываем, сколько раз встречается каждый order_id
order_counts = client_orders["order_id"].value_counts()

# Оставляем ID, которые встречаются больше одного раза
repeated_order_id = order_counts[order_counts > 1]

print(repeated_order_id)
print("\nСтроки с этим заказом:")
print(client_orders.loc[client_orders["order_id"].isin(repeated_order_id.index)])

Клиент сделавщий больше всего заказов:
C01818    11
Name: customer_id, dtype: int64

Все заказы этого клиента:
      order_id        order_date       city   amount   status
57      101125  2026-09-01 15:00    Шымкент   7350.0  оплачен
779     100096  2026-09-09 18:00    Шымкент  45270.0  оплачен
1371    101504  2026-09-15 21:00     Алматы   8890.0  оплачен
1375    101504  2026-09-15 21:00     Алматы   8890.0  оплачен
2195    101860  2026-09-24 18:00     Астана   1490.0  оплачен
2617    101462  2026-09-29 09:00     Астана  22170.0  оплачен
2776    101495  2026-09-30 21:00     Алматы  20180.0  оплачен
3417    103416  2026-10-08 14:00  Караганда   4330.0  оплачен
4279    102644  2026-10-18 08:00     Астана  13600.0  оплачен
4610    104001  2026-10-21 19:00     Алматы  21990.0  оплачен
4778    104572  2026-10-23 18:00     Алматы  11140.0  оплачен

101504    2
Name: order_id, dtype: int64

Строки с этим заказом:
      order_id        order_date    city  amount   status
1371    101504  2026-

**Вывод:**
- Клиент с наибольшим количеством заказов — C01818, у него 11 записей. При проверке обнаружен повторяющийся order_id = 101504: один и тот же оплаченный заказ на 8 890 ₸ присутствует дважды.
- Это означает, что при расчёте выручки без удаления дубликатов один и тот же заказ учитывается несколько раз. Поэтому выручка может быть завышена. Перед окончательным финансовым анализом необходимо проверить и удалить полные дубликаты, чтобы избежать двойного учёта.

## [Челлендж] 7. Выбросы по правилу IQR
Для оплаченных заказов без TEST: `IQR = Q3 − Q1`, граница = `Q3 + 1.5 * IQR`. Сколько заказов выше границы, какая это доля и какая у них категория чаще всего?
Сравните с ДЗ 4: по z-оценке выбросов было 87. Почему правила дают разное число?

In [29]:
# 7
Q1 = paid_not_test["amount"].quantile(0.25)
Q3 = paid_not_test["amount"].quantile(0.75)

IQR = Q3 - Q1
border = Q3 + 1.5 * IQR

outliers = paid_not_test[paid_not_test["amount"] > border]

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Граница:", border)

print("Количество выбросов:", len(outliers))
print("Доля выбросов:", round(len(outliers) / len(paid_not_test) * 100, 2), "%")

print(outliers["category"].value_counts())

Q1: 5880.0
Q3: 18690.0
IQR: 12810.0
Граница: 37905.0
Количество выбросов: 320
Доля выбросов: 6.53 %
Электроника    308
Одежда          10
Дом              2
Name: category, dtype: int64


**Выводы:**
- Чаще всего выбросы встречаются в категории «Электроника» — 308 заказов из 320.
- Количество выбросов в Z-оценка (ДЗ 4) = 87, а в IQR = 320. IQR выявил на 233 выброса больше.
- Z-оценка оценивает, насколько далеко значение находится от среднего в единицах стандартного отклонения. При пороге 3 стандартных отклонения метод выделяет только наиболее экстремальные значения.
- IQR использует медианное положение данных через квартиль Q1 и Q3. Он менее чувствителен к крупным значениям и в нашем случае установил более низкую границу — 37 905 ₸.
- Разница объясняется тем, что методы используют разные принципы определения выбросов. IQR в данном случае имеет более низкую границу и обнаруживает больше крупных заказов.

## [Челлендж] 8. Доля отмен для каждого написания города одной строкой
Поделите две Series: число отменённых заказов по `city` и число всех заказов по `city` (обе через `value_counts`). Отсортируйте.
Pandas делит по ярлыкам: проверьте это. Доля отмен у «Шымкент», «шымкент» и «Shymkent» сильно разная. Сколько заказов у каждого написания?
Можно ли судить о городе по одному написанию? Что нужно сделать с городами до анализа (это тема урока 6)?

In [35]:
# 8
cancel_share = (df.loc[df["status"] == "отменён", "city"].value_counts() / df["city"].value_counts()).sort_values()

print(cancel_share * 100)

cancel_counts = df.loc[df["status"] == "отменён", "city"].value_counts()
all_counts = df["city"].value_counts()

print("\nВсего заказов:")
print(all_counts.loc[["Шымкент", "шымкент", "Shymkent"]])

print("\nОтменённые заказы:")
print(cancel_counts.loc[["Шымкент", "шымкент", "Shymkent"]])

print("\nДоля отмен, %:")
print((cancel_counts / all_counts * 100).loc[
    ["Шымкент", "шымкент", "Shymkent"]
])

АЛМАТЫ        3.225806
шымкент       3.703704
 Алматы       4.545455
Almaty        5.797101
Атырау        7.534247
Алматы        7.991697
Астана        8.482871
Караганда     8.921933
Astana       10.000000
Актобе       10.327456
астана       13.095238
алматы       14.062500
Шымкент      21.518987
Shymkent     23.076923
Name: city, dtype: float64

Всего заказов:
Шымкент     711
шымкент      54
Shymkent     39
Name: city, dtype: int64

Отменённые заказы:
Шымкент     153
шымкент       2
Shymkent      9
Name: city, dtype: int64

Доля отмен, %:
Шымкент     21.518987
шымкент      3.703704
Shymkent    23.076923
Name: city, dtype: float64
